# Imports

In [10]:
#imports
import jax
import jax.numpy as jnp
import flax.linen as nn
import optax
import numpy as np
from datasets import load_dataset
from transformers import AutoTokenizer
from tree_sitter import Language, Parser
import tree_sitter_python as tspython
from scipy.stats import spearmanr
import matplotlib.pyplot as plt

/usr/local/lib/python3.12/site-packages/jax/_src/cloud_tpu_init.py:93: UserWarning: Transparent hugepages are not enabled. TPU runtime startup and shutdown time should be significantly improved on TPU v5e and newer. If not already set, you may need to enable transparent hugepages in your VM image (sudo sh -c "echo always > /sys/kernel/mm/transparent_hugepage/enabled")
  warnings.warn(
/usr/local/lib/python3.12/site-packages/torch_xla/__init__.py:258: UserWarning: `tensorflow` can conflict with `torch-xla`. Prefer `tensorflow-cpu` when using PyTorch/XLA. To silence this warning, `pip uninstall -y tensorflow && pip install tensorflow-cpu`. If you are in a notebook environment such as Colab or Kaggle, restart your notebook runtime afterwards.
  warnings.warn(


# Hugging Face Token Authentication 

In [12]:
import os
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

secret_value = UserSecretsClient().get_secret("HF_TOKEN")
login(token=secret_value)

In [13]:
#fetch the dataset 
dataset = load_dataset(
    "bigcode/the-stack",
    data_dir="data/python",
    split="train",
    streaming=True
)
print("Dataset ready")

Resolving data files:   0%|          | 0/206 [00:00<?, ?it/s]

Dataset ready


# Cleaing the Notebook for Starting again if you face some error 

In [ ]:
# Clear everything
import gc
import jax

# Delete existing variables
try:
    del model, params, opt_state, grads
    del batch, dataset_iter
except:
    pass

# Force garbage collection
gc.collect()

# Clear JAX cache
jax.clear_caches()

print("Cleared")

# Parser And Tokenizer

In [14]:
# Build parser
PY_LANGUAGE = Language(tspython.language(), "python")
parser = Parser()
parser.set_language(PY_LANGUAGE)

# Load tokenizer once — reuse everywhere
tokenizer = AutoTokenizer.from_pretrained("bigcode/starcoder2-3b")

def parse_code(source):
    try:
        tree = parser.parse(bytes(source, "utf8"))
        if tree.root_node.has_error:
            return None
        return tree
    except:
        return None

def extract_ast_edges(tree):
    nodes, edges = [], []
    node_id = [0]

    def traverse(node, parent_id=None):
        current_id = node_id[0]
        node_id[0] += 1
        nodes.append({
            "id":         current_id,
            "type":       node.type,
            "start_byte": node.start_byte,
            "end_byte":   node.end_byte,
        })
        if parent_id is not None:
            edges.append((parent_id, current_id))
        for child in node.children:
            traverse(child, current_id)

    traverse(tree.root_node)

    # Compute depths via BFS
    depth_map = {0: 0}
    for parent_id, child_id in edges:
        depth_map[child_id] = depth_map.get(parent_id, 0) + 1
    for node in nodes:
        node["depth"] = depth_map.get(node["id"], 0)

    return nodes, edges

def align_tokens_to_ast(source, nodes):
    encoding = tokenizer(
        source,
        return_offsets_mapping=True,
        truncation=True,
        max_length=512
    )
    token_ids  = encoding["input_ids"]
    offset_map = encoding["offset_mapping"]

    token_node_map = []
    for (char_start, char_end) in offset_map:
        assigned = 0
        for node in reversed(nodes):
            if (node["start_byte"] <= char_start and
                node["end_byte"]   >= char_end):
                assigned = node["id"]
                break
        token_node_map.append(assigned)

    depths = [
        nodes[nid]["depth"] if nid < len(nodes) else 0
        for nid in token_node_map
    ]

    return token_ids, token_node_map, depths

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/958 [00:00<?, ?B/s]

# Test Pipeline

In [ ]:
dataset = load_dataset(
    "bigcode/the-stack",
    data_dir="data/python",
    split="train",
    streaming=True
)

passed, failed = 0, 0
sample_output  = None

for i, example in enumerate(dataset):
    if i >= 100:
        break
    source = example["content"]
    if len(source) > 50_000:
        continue

    tree = parse_code(source)
    if tree is None:
        failed += 1
        continue

    nodes, edges = extract_ast_edges(tree)
    token_ids, token_node_map, depths = align_tokens_to_ast(
        source, nodes)

    if sample_output is None:
        sample_output = {
            "token_ids":      token_ids[:10],
            "token_node_map": token_node_map[:10],
            "depths":         depths[:10],
            "n_nodes":        len(nodes),
            "n_edges":        len(edges),
        }
    passed += 1

print(f"Pass rate: {passed}/{passed+failed}")
print(f"\nSample output:")
for k, v in sample_output.items():
    print(f"  {k}: {v}")

# Geometry Sanity Check

In [ ]:
def embed_on_hyperboloid(token_ids, depths, dim=32):
    np.random.seed(42)
    vocab_size = tokenizer.vocab_size
    E          = np.random.randn(vocab_size, dim) * 0.1
    spatial    = np.array([E[t] for t in token_ids])
    scale      = np.exp(np.array(depths, dtype=float) * 0.1)
    spatial    = spatial * scale[:, None]
    spatial    = np.clip(spatial, -8.0, 8.0)
    x0         = np.sqrt(1 + np.sum(spatial**2,
                                     axis=-1, keepdims=True))
    return np.concatenate([x0, spatial], axis=-1)

def bfs_distance(adj, start, end):
    if start == end:
        return 0
    visited = {start}
    queue   = [(start, 0)]
    while queue:
        node, dist = queue.pop(0)
        for neighbor in adj.get(node, []):
            if neighbor == end:
                return dist + 1
            if neighbor not in visited:
                visited.add(neighbor)
                queue.append((neighbor, dist + 1))
    return -1

def lorentzian_inner_np(x, y):
    return (-x[0] * y[0] +
             np.sum(x[1:] * y[1:]))

def hyperbolic_dist_np(x, y):
    inner = np.clip(-lorentzian_inner_np(x, y),
                    1 + 1e-6, 1e8)
    return np.arccosh(inner)

# Run on 10 files
hop_dists, hyp_dists = [], []
dataset_iter = iter(dataset)
files_done   = 0

while files_done < 10:
    example = next(dataset_iter)
    source  = example["content"]
    if len(source) > 10_000:
        continue

    tree = parse_code(source)
    if tree is None:
        continue

    nodes, edges = extract_ast_edges(tree)
    if len(nodes) < 10:
        continue

    token_ids, token_node_map, depths = align_tokens_to_ast(
        source, nodes)
    embeddings = embed_on_hyperboloid(token_ids, depths)

    # Build adjacency
    adj = {}
    for p, c in edges:
        adj.setdefault(p, []).append(c)
        adj.setdefault(c, []).append(p)

    # Sample 50 pairs per file
    node_ids = list(range(len(nodes)))
    for _ in range(50):
        a = np.random.choice(node_ids)
        b = np.random.choice(node_ids)
        d = bfs_distance(adj, a, b)
        if d < 0:
            continue

        tok_a = next((i for i, n in
                      enumerate(token_node_map) if n == a), None)
        tok_b = next((i for i, n in
                      enumerate(token_node_map) if n == b), None)

        if tok_a is not None and tok_b is not None:
            h = hyperbolic_dist_np(embeddings[tok_a],
                                    embeddings[tok_b])
            hop_dists.append(d)
            hyp_dists.append(h)

    files_done += 1

corr, p = spearmanr(hyp_dists, hop_dists)
print(f"Spearman correlation: {corr:.3f}  (p={p:.4f})")

if corr > 0.5:
    print("Strong")
elif corr > 0.2:
    print("Weak")
else:
    print("Failed")

# Geometry Sanity Check Result

**Date:** May 3, 2026  
**Configuration:**
- depth_scale = 0.1
- dim = 32
- files = 10
- pairs = 500 (50 per file)

**Result:** Spearman correlation = 0.703 (p < 0.0001)

Before any training, hyperbolic distance already reflects AST 
structure with 70% rank correlation. Tokens close in the tree 
are close in hyperbolic space and tokens far in the tree are 
far in hyperbolic space. The geometry is working from depth 
to radius mapping alone, with no learned parameters.

This confirms the core hypothesis that code is fundamentally hierarchical, and hyperbolic space is the natural geometry for hierarchical data. A transformer trained with hyperbolic attention and AST-aligned tokenization should encode code structure geometrically rather than having to learn it statistically from flat token sequences. This means the model can use its full parameter capacity for semantic understanding rather than spending it rediscovering structure that the geometry already provides for free. is geometrically sound. Proceeding to model training.

# Model Definition in JAX

In [ ]:
class HyperbolicAttention(nn.Module):
    num_heads: int
    head_dim:  int

    @nn.compact
    def __call__(self, x):
        B, S, D = x.shape
        H, Hd   = self.num_heads, self.head_dim

        Q = nn.Dense(H * Hd)(x).reshape(B, S, H, Hd).transpose(0, 2, 1, 3)
        K = nn.Dense(H * Hd)(x).reshape(B, S, H, Hd).transpose(0, 2, 1, 3)
        V = nn.Dense(H * Hd)(x).reshape(B, S, H, Hd).transpose(0, 2, 1, 3)

        # Project Q, K onto hyperboloid
        def to_hyperboloid(z):
            z = jnp.clip(z, -8.0, 8.0)
            z0 = jnp.sqrt(1 + jnp.sum(z**2,
                                        axis=-1, keepdims=True))
            return jnp.concatenate([z0, z], axis=-1)

        Q_h = to_hyperboloid(Q)
        K_h = to_hyperboloid(K)

        # Approximate hyperbolic distance which is TPU friendly
        Q_s = Q_h[..., 1:] / jnp.maximum(
            jnp.linalg.norm(Q_h[..., 1:], axis=-1, keepdims=True), 1e-8)
        K_s = K_h[..., 1:] / jnp.maximum(
            jnp.linalg.norm(K_h[..., 1:], axis=-1, keepdims=True), 1e-8)

        cos_sim = jnp.einsum("bhid,bhjd->bhij", Q_s, K_s)
        scale   = (Q_h[..., 0:1] *
                   K_h[..., 0:1].transpose(0, 1, 3, 2))
        scores  = -scale * (1 - cos_sim)

        # Learnable temperature
        log_t  = self.param('log_temp',
                              nn.initializers.zeros, ())
        temp   = jnp.maximum(jnp.exp(log_t), 0.1)
        scores = scores / temp

        # Stable softmax
        scores  = scores - jnp.max(scores,
                                    axis=-1, keepdims=True)
        weights = jax.nn.softmax(scores, axis=-1)

        out = jnp.einsum("bhij,bhjd->bhid", weights, V)
        out = out.transpose(0, 2, 1, 3).reshape(B, S, H * Hd)
        return nn.Dense(D)(out)


class HyperbolicCodingModel(nn.Module):
    vocab_size: int = tokenizer.vocab_size
    hidden_dim: int = 16
    num_layers: int = 4
    num_heads:  int = 4
    max_depth:  int = 20

    @nn.compact
    def __call__(self, token_ids, depths):
        # Embeddings
        tok_emb = nn.Embed(self.vocab_size,
                            self.hidden_dim)(token_ids)
        dep_emb = nn.Embed(self.max_depth,
                            self.hidden_dim)(
                                jnp.clip(depths, 0,
                                         self.max_depth - 1))
        x = tok_emb + dep_emb

        # Project onto hyperboloid using depth as radius
        spatial   = x / jnp.maximum(
            jnp.linalg.norm(x, axis=-1, keepdims=True), 1e-8)
        depth_f   = depths.astype(jnp.float32)
        scale     = jnp.exp(depth_f * 0.1)[..., None]
        spatial   = jnp.clip(spatial * scale, -8.0, 8.0)

        # Transformer layers
        for _ in range(self.num_layers):
            residual = spatial
            attn_out = HyperbolicAttention(
                self.num_heads,
                self.hidden_dim // self.num_heads
            )(spatial)
            spatial = nn.LayerNorm()(residual + attn_out)

            ffn     = nn.Dense(self.hidden_dim * 4)(spatial)
            ffn     = nn.gelu(ffn)
            ffn     = nn.Dense(self.hidden_dim)(ffn)
            spatial = nn.LayerNorm()(spatial + ffn)
            spatial = jnp.clip(spatial, -8.0, 8.0)

        return nn.Dense(self.vocab_size)(spatial)

# Verify Model on CPU

In [ ]:
model = HyperbolicCodingModel()
key   = jax.random.PRNGKey(0)

dummy_tokens = jnp.ones((2, 64), dtype=jnp.int32)
dummy_depths = jnp.ones((2, 64), dtype=jnp.int32)

params = model.init(key, dummy_tokens, dummy_depths)

# Count params
n_params = sum(x.size for x in
               jax.tree_util.tree_leaves(params))
print(f"Parameters: {n_params/1e6:.2f}M")

# Forward pass
logits = model.apply(params, dummy_tokens, dummy_depths)
print(f"Output shape: {logits.shape}")
# Expected: (2, 64, 32000)

# Backward pass
def loss_fn(params):
    logits  = model.apply(params, dummy_tokens, dummy_depths)
    targets = dummy_tokens[:, 1:]
    logits  = logits[:, :-1, :]
    return optax.softmax_cross_entropy_with_integer_labels(
        logits, targets).mean()

loss, grads = jax.value_and_grad(loss_fn)(params)
print(f"Loss: {loss:.4f}")
print(f"Gradients flowing: "
      f"{not any(jnp.any(jnp.isnan(g)) for g in jax.tree_util.tree_leaves(grads))}")

# Verify Training loop works on CPU

In [ ]:
def make_batch(dataset_iter, batch_size=2, seq_len=64):
    """Pull one batch from the stream"""
    batch_tokens = []
    batch_depths = []

    while len(batch_tokens) < batch_size:
        try:
            example = next(dataset_iter)
        except StopIteration:
            break

        source = example["content"]
        if len(source) > 50_000:
            continue

        tree = parse_code(source)
        if tree is None:
            continue

        nodes, edges = extract_ast_edges(tree)
        token_ids, _, depths = align_tokens_to_ast(source, nodes)

        # Pad or truncate to seq_len
        token_ids = token_ids[:seq_len]
        depths    = depths[:seq_len]

        pad_len   = seq_len - len(token_ids)
        token_ids = token_ids + [0] * pad_len
        depths    = depths    + [0] * pad_len

        batch_tokens.append(token_ids)
        batch_depths.append(depths)

    if len(batch_tokens) < batch_size:
        return None

    return {
        "token_ids": jnp.array(batch_tokens, dtype=jnp.int32),
        "depths":    jnp.array(batch_depths,  dtype=jnp.int32),
    }

# Test training loop (10 steps on a CPU)
optimizer = optax.adam(1e-3)
opt_state = optimizer.init(params)

@jax.jit
def train_step(params, opt_state, batch):
    def loss_fn(params):
        logits  = model.apply(params,
                               batch["token_ids"],
                               batch["depths"])
        targets = batch["token_ids"][:, 1:]
        logits  = logits[:, :-1, :]
        return optax.softmax_cross_entropy_with_integer_labels(
            logits, targets).mean()

    loss, grads = jax.value_and_grad(loss_fn)(params)
    updates, new_opt_state = optimizer.update(grads, opt_state)
    new_params = optax.apply_updates(params, updates)
    return loss, new_params, new_opt_state

dataset_iter = iter(dataset)

print("Running 10 test steps on CPU...")
for step in range(10):
    batch = make_batch(dataset_iter, batch_size=2, seq_len=64)
    if batch is None:
        break

    loss, params, opt_state = train_step(params,
                                          opt_state, batch)
    print(f"Step {step}: loss = {loss:.4f}")

# Debugging Gradient Explosion 

Got nan on the first step itself , have to write a dubugging script for step by step reproduction 

In [ ]:
#Reinitializing params because training loop currupted them and caused Nan Explosion on the first training step itself  

# Reinitialize completely fresh
model  = HyperbolicCodingModel()
key    = jax.random.PRNGKey(0)

dummy_tokens = jnp.ones((2, 64), dtype=jnp.int32)
dummy_depths = jnp.ones((2, 64), dtype=jnp.int32)

params = model.init(key, dummy_tokens, dummy_depths)

# Verify embeddings are clean
emb = params['params']['Embed_0']['embedding']
print("Embedding has NaN:", jnp.any(jnp.isnan(emb)))
print("Embedding min/max:", emb.min(), emb.max())

# Get one batch
dataset_iter = iter(dataset)
batch = make_batch(dataset_iter, batch_size=2, seq_len=64)

print("depths min/max:", batch["depths"].min(), batch["depths"].max())
print("token_ids min/max:", batch["token_ids"].min(), batch["token_ids"].max())

# Step through forward pass using model.apply with intermediates
def debug_forward(params, token_ids, depths):
    # Step 1 : embeddings
    tok_emb = params['params']['Embed_0']['embedding'][token_ids]
    dep_emb = params['params']['Embed_1']['embedding'][jnp.clip(depths, 0, 19)]
    x = tok_emb + dep_emb
    print("x has NaN:", jnp.any(jnp.isnan(x)))
    print("x norm mean:", jnp.linalg.norm(x, axis=-1).mean())

    # Step 2 : normalize
    spatial = x / jnp.maximum(
        jnp.linalg.norm(x, axis=-1, keepdims=True), 1e-8)
    print("spatial has NaN:", jnp.any(jnp.isnan(spatial)))

    # Step 3 : depth scaling
    depth_f = depths.astype(jnp.float32)
    scale   = jnp.exp(depth_f * 0.1)[..., None]
    print("scale min/max:", scale.min(), scale.max())
    spatial = jnp.clip(spatial * scale, -8.0, 8.0)
    print("spatial after scale has NaN:", jnp.any(jnp.isnan(spatial)))

    # Step 4 : full forward pass
    logits = model.apply(params, token_ids, depths)
    print("logits has NaN:", jnp.any(jnp.isnan(logits)))
    print("logits min/max:", logits.min(), logits.max())

debug_forward(params, batch["token_ids"], batch["depths"])

Adding debug to the Attention layer , because that is where the earlier debug script isolated the Nan explosion to 

In [ ]:
def debug_attention(params, token_ids, depths):
    # Reproduce the model forward manually up to attention
    tok_emb = params['params']['Embed_0']['embedding'][token_ids]
    dep_emb = params['params']['Embed_1']['embedding'][jnp.clip(depths, 0, 19)]
    x = tok_emb + dep_emb

    spatial = x / jnp.maximum(
        jnp.linalg.norm(x, axis=-1, keepdims=True), 1e-8)
    depth_f = depths.astype(jnp.float32)
    scale   = jnp.exp(depth_f * 0.1)[..., None]
    spatial = jnp.clip(spatial * scale, -8.0, 8.0)

    print("spatial into attention has NaN:", jnp.any(jnp.isnan(spatial)))
    print("spatial norm mean:", jnp.linalg.norm(spatial, axis=-1).mean())

    # Simulate to_hyperboloid
    x0  = jnp.sqrt(1 + jnp.sum(spatial**2, axis=-1, keepdims=True))
    x_h = jnp.concatenate([x0, spatial], axis=-1)
    print("x_h has NaN:", jnp.any(jnp.isnan(x_h)))
    print("x0 min/max:", x0.min(), x0.max())

    # Simulate Q projection and get weights from params
    # Layer 0 attention dense weights
    q_kernel = params['params']['HyperbolicAttention_0']['Dense_0']['kernel']
    B, S, D  = spatial.shape
    H, Hd    = 4, 4  # num_heads, head_dim
    Q = jnp.dot(spatial, q_kernel).reshape(B, S, H, Hd).transpose(0, 2, 1, 3)
    print("Q has NaN:", jnp.any(jnp.isnan(Q)))

    # to_hyperboloid on Q
    Q   = jnp.clip(Q, -8.0, 8.0)
    Q_0 = jnp.sqrt(1 + jnp.sum(Q**2, axis=-1, keepdims=True))
    Q_h = jnp.concatenate([Q_0, Q], axis=-1)
    print("Q_h has NaN:", jnp.any(jnp.isnan(Q_h)))

    # Normalize spatial part
    Q_s = Q_h[..., 1:] / jnp.maximum(
        jnp.linalg.norm(Q_h[..., 1:], axis=-1, keepdims=True), 1e-8)
    print("Q_s has NaN:", jnp.any(jnp.isnan(Q_s)))

    # Same for K
    k_kernel = params['params']['HyperbolicAttention_0']['Dense_1']['kernel']
    K = jnp.dot(spatial, k_kernel).reshape(B, S, H, Hd).transpose(0, 2, 1, 3)
    K   = jnp.clip(K, -8.0, 8.0)
    K_0 = jnp.sqrt(1 + jnp.sum(K**2, axis=-1, keepdims=True))
    K_h = jnp.concatenate([K_0, K], axis=-1)
    K_s = K_h[..., 1:] / jnp.maximum(
        jnp.linalg.norm(K_h[..., 1:], axis=-1, keepdims=True), 1e-8)
    print("K_s has NaN:", jnp.any(jnp.isnan(K_s)))

    # Cosine similarity
    cos_sim = jnp.einsum("bhid,bhjd->bhij", Q_s, K_s)
    print("cos_sim has NaN:", jnp.any(jnp.isnan(cos_sim)))
    print("cos_sim min/max:", cos_sim.min(), cos_sim.max())

    # Scale term
    scale_term = Q_h[..., 0:1] * K_h[..., 0:1].transpose(0, 1, 3, 2)
    print("scale_term has NaN:", jnp.any(jnp.isnan(scale_term)))
    print("scale_term min/max:", scale_term.min(), scale_term.max())

    # Scores
    scores = -scale_term * (1 - cos_sim)
    print("scores has NaN:", jnp.any(jnp.isnan(scores)))

debug_attention(params, batch["token_ids"], batch["depths"])

# Dubugging into furthur layers 

An Intersting finding from the debug of the first attention layer is that , the Nan does not appear immediately , it appears in deeper , layers so now i have made a script to go deeper into the model and see when does the Nan appear , and what are the values before it to see emperical relationship 

In [ ]:
def debug_all_layers(params, token_ids, depths):
    
    # Initial embedding
    tok_emb = params['params']['Embed_0']['embedding'][token_ids]
    dep_emb = params['params']['Embed_1']['embedding'][
        jnp.clip(depths, 0, 19)]
    x = tok_emb + dep_emb

    spatial = x / jnp.maximum(
        jnp.linalg.norm(x, axis=-1, keepdims=True), 1e-8)
    depth_f = depths.astype(jnp.float32)
    scale   = jnp.exp(depth_f * 0.1)[..., None]
    spatial = jnp.clip(spatial * scale, -8.0, 8.0)

    print("=" * 55)
    print(f"{'Stage':<30} {'NaN':>5} {'Min':>8} {'Max':>8} {'Mean':>8}")
    print("=" * 55)

    def log(name, tensor):
        has_nan = bool(jnp.any(jnp.isnan(tensor)))
        mn      = float(jnp.nanmin(tensor))
        mx      = float(jnp.nanmax(tensor))
        mean    = float(jnp.nanmean(tensor))
        flag    = "YES" if has_nan else "no"
        print(f"{name:<30} {flag:>5} {mn:>8.4f} {mx:>8.4f} {mean:>8.4f}")
        return has_nan

    log("input spatial", spatial)

    for layer_idx in range(4):  # num_layers = 4
        prefix = f"HyperbolicAttention_{layer_idx}"

        # Q, K, V projections
        q_w = params['params'][prefix]['Dense_0']['kernel']
        k_w = params['params'][prefix]['Dense_1']['kernel']
        v_w = params['params'][prefix]['Dense_2']['kernel']
        o_w = params['params'][prefix]['Dense_3']['kernel']

        B, S, D = spatial.shape
        H, Hd   = 4, D // 4

        Q = jnp.dot(spatial, q_w).reshape(B, S, H, Hd).transpose(0,2,1,3)
        K = jnp.dot(spatial, k_w).reshape(B, S, H, Hd).transpose(0,2,1,3)
        V = jnp.dot(spatial, v_w).reshape(B, S, H, Hd).transpose(0,2,1,3)

        log(f"layer{layer_idx} Q", Q)
        log(f"layer{layer_idx} K", K)
        log(f"layer{layer_idx} V", V)

        # Hyperboloid projection
        Q   = jnp.clip(Q, -8.0, 8.0)
        K   = jnp.clip(K, -8.0, 8.0)
        Q_0 = jnp.sqrt(1 + jnp.sum(Q**2, axis=-1, keepdims=True))
        K_0 = jnp.sqrt(1 + jnp.sum(K**2, axis=-1, keepdims=True))
        Q_h = jnp.concatenate([Q_0, Q], axis=-1)
        K_h = jnp.concatenate([K_0, K], axis=-1)

        log(f"layer{layer_idx} Q_h x0", Q_0)
        log(f"layer{layer_idx} K_h x0", K_0)

        # Normalize spatial
        Q_s = Q_h[..., 1:] / jnp.maximum(
            jnp.linalg.norm(Q_h[..., 1:], axis=-1, keepdims=True), 1e-8)
        K_s = K_h[..., 1:] / jnp.maximum(
            jnp.linalg.norm(K_h[..., 1:], axis=-1, keepdims=True), 1e-8)

        log(f"layer{layer_idx} Q_s", Q_s)
        log(f"layer{layer_idx} K_s", K_s)

        # Scores
        cos_sim    = jnp.einsum("bhid,bhjd->bhij", Q_s, K_s)
        scale_term = (Q_h[..., 0:1] *
                      K_h[..., 0:1].transpose(0, 1, 3, 2))
        scores     = -scale_term * (1 - cos_sim)

        log(f"layer{layer_idx} cos_sim", cos_sim)
        log(f"layer{layer_idx} scale_term", scale_term)
        log(f"layer{layer_idx} scores", scores)

        # Temperature
        log_t  = params['params'][prefix]['log_temp']
        temp   = jnp.maximum(jnp.exp(log_t), 0.1)
        scores = scores / temp
        log(f"layer{layer_idx} scores/temp", scores)

        # Softmax
        scores  = scores - jnp.max(scores, axis=-1, keepdims=True)
        weights = jax.nn.softmax(scores, axis=-1)
        log(f"layer{layer_idx} weights", weights)

        # Weighted sum
        out = jnp.einsum("bhij,bhjd->bhid", weights, V)
        out = out.transpose(0, 2, 1, 3).reshape(B, S, H * Hd)
        out = jnp.dot(out, o_w)
        log(f"layer{layer_idx} attn_out", out)

        # Residual + LayerNorm
        residual = spatial
        spatial  = spatial + out
        log(f"layer{layer_idx} after residual", spatial)

        # LayerNorm : get params
        ln1_scale = params['params'][f'LayerNorm_{layer_idx*2}']['scale']
        ln1_bias  = params['params'][f'LayerNorm_{layer_idx*2}']['bias']
        mean_s    = jnp.mean(spatial, axis=-1, keepdims=True)
        std_s     = jnp.std(spatial,  axis=-1, keepdims=True)
        spatial   = (spatial - mean_s) / jnp.maximum(std_s, 1e-6)
        spatial   = spatial * ln1_scale + ln1_bias
        log(f"layer{layer_idx} after layernorm1", spatial)
        spatial   = jnp.clip(spatial, -8.0, 8.0)

        # FFN
        ffn_w1 = params['params'][f'Dense_{layer_idx*2}']['kernel']
        ffn_b1 = params['params'][f'Dense_{layer_idx*2}']['bias']
        ffn_w2 = params['params'][f'Dense_{layer_idx*2+1}']['kernel']
        ffn_b2 = params['params'][f'Dense_{layer_idx*2+1}']['bias']

        ffn = jnp.dot(spatial, ffn_w1) + ffn_b1
        log(f"layer{layer_idx} ffn pre-gelu", ffn)
        ffn = jax.nn.gelu(ffn)
        ffn = jnp.dot(ffn, ffn_w2) + ffn_b2
        log(f"layer{layer_idx} ffn output", ffn)

        spatial = spatial + ffn
        log(f"layer{layer_idx} after ffn residual", spatial)

        ln2_scale = params['params'][f'LayerNorm_{layer_idx*2+1}']['scale']
        ln2_bias  = params['params'][f'LayerNorm_{layer_idx*2+1}']['bias']
        mean_s    = jnp.mean(spatial, axis=-1, keepdims=True)
        std_s     = jnp.std(spatial,  axis=-1, keepdims=True)
        spatial   = (spatial - mean_s) / jnp.maximum(std_s, 1e-6)
        spatial   = spatial * ln2_scale + ln2_bias
        log(f"layer{layer_idx} after layernorm2", spatial)
        spatial   = jnp.clip(spatial, -8.0, 8.0)

        print("-" * 55)

    # Final output projection
    out_w   = params['params']['Dense_0']['kernel']
    out_b   = params['params']['Dense_0']['bias']
    logits  = jnp.dot(spatial, out_w) + out_b
    log("final logits", logits)
    print("=" * 55)

debug_all_layers(params, batch["token_ids"], batch["depths"])

# Interesting Findings 

So the debug script for each individual layer proved , that the issue is not with the activations , the activations are propagating just fine, i.e. the forward step is fine , but the gradients in the backward step most probably explore , so Now I am going to write a script for debugging the backward step 

# Backward step Debugging 

In [ ]:
def debug_gradients(params, token_ids, depths):
    
    def loss_fn(params):
        logits  = model.apply(params, token_ids, depths)
        targets = token_ids[:, 1:]
        logits  = logits[:, :-1, :]
        return optax.softmax_cross_entropy_with_integer_labels(
            logits, targets).mean()

    # Compute gradients
    loss, grads = jax.value_and_grad(loss_fn)(params)
    
    print(f"Loss: {loss:.4f}")
    print()
    print("=" * 65)
    print(f"{'Parameter':<40} {'NaN':>5} {'Min':>8} {'Max':>8} {'Mean':>8}")
    print("=" * 65)

    def log_grad(name, g):
        has_nan = bool(jnp.any(jnp.isnan(g)))
        mn      = float(jnp.nanmin(jnp.abs(g)))
        mx      = float(jnp.nanmax(jnp.abs(g)))
        mean    = float(jnp.nanmean(jnp.abs(g)))
        flag    = "YES" if has_nan else "no"
        print(f"{name:<40} {flag:>5} {mn:>8.6f} {mx:>8.4f} {mean:>8.6f}")
        return has_nan

    # Embeddings
    log_grad("Embed_0 (token)",
             grads['params']['Embed_0']['embedding'])
    log_grad("Embed_1 (depth)",
             grads['params']['Embed_1']['embedding'])
    print("-" * 65)

    # Each layer
    for layer_idx in range(4):
        prefix = f"HyperbolicAttention_{layer_idx}"

        log_grad(f"layer{layer_idx} Q kernel",
                 grads['params'][prefix]['Dense_0']['kernel'])
        log_grad(f"layer{layer_idx} K kernel",
                 grads['params'][prefix]['Dense_1']['kernel'])
        log_grad(f"layer{layer_idx} V kernel",
                 grads['params'][prefix]['Dense_2']['kernel'])
        log_grad(f"layer{layer_idx} O kernel",
                 grads['params'][prefix]['Dense_3']['kernel'])
        log_grad(f"layer{layer_idx} log_temp",
                 grads['params'][prefix]['log_temp'])

        # LayerNorms
        ln1 = f"LayerNorm_{layer_idx*2}"
        ln2 = f"LayerNorm_{layer_idx*2+1}"
        log_grad(f"layer{layer_idx} LN1 scale",
                 grads['params'][ln1]['scale'])
        log_grad(f"layer{layer_idx} LN1 bias",
                 grads['params'][ln1]['bias'])
        log_grad(f"layer{layer_idx} LN2 scale",
                 grads['params'][ln2]['scale'])
        log_grad(f"layer{layer_idx} LN2 bias",
                 grads['params'][ln2]['bias'])

        # FFN
        log_grad(f"layer{layer_idx} FFN W1",
                 grads['params'][f'Dense_{layer_idx*2}']['kernel'])
        log_grad(f"layer{layer_idx} FFN W2",
                 grads['params'][f'Dense_{layer_idx*2+1}']['kernel'])
        print("-" * 65)

    # Output projection
    log_grad("output Dense kernel",
             grads['params']['Dense_0']['kernel'])
    log_grad("output Dense bias",
             grads['params']['Dense_0']['bias'])

    print("=" * 65)

    # Summary : which layers have the largest gradient norms
    print("\nGradient norm per layer (useful for spotting explosion):")
    print("-" * 45)
    for layer_idx in range(4):
        prefix = f"HyperbolicAttention_{layer_idx}"
        layer_grads = [
            grads['params'][prefix]['Dense_0']['kernel'],
            grads['params'][prefix]['Dense_1']['kernel'],
            grads['params'][prefix]['Dense_2']['kernel'],
            grads['params'][prefix]['Dense_3']['kernel'],
        ]
        layer_norm = jnp.sqrt(sum(
            jnp.sum(g**2) for g in layer_grads
        ))
        print(f"  layer{layer_idx} attention grad norm: {layer_norm:.4f}")

    ffn_norms = []
    for layer_idx in range(4):
        ffn_grads = [
            grads['params'][f'Dense_{layer_idx*2}']['kernel'],
            grads['params'][f'Dense_{layer_idx*2+1}']['kernel'],
        ]
        ffn_norm = jnp.sqrt(sum(
            jnp.sum(g**2) for g in ffn_grads
        ))
        ffn_norms.append(ffn_norm)
        print(f"  layer{layer_idx} FFN grad norm:       {ffn_norm:.4f}")

debug_gradients(params, batch["token_ids"], batch["depths"])

# Error Found 
The error found was pretty mundane . Through the systemic dubugging process , the error was found in the vocabulary size . Earlier , the size was set to a constant value 32000 , but the tokenizer used a Vocab size was 49152 . So , some of the targets while backpropagation were out of bound of the tokenizer , so the loss value was Nan , and then that propagated through . Thus , neither the backward step , nor the forward step was wrong . 

# TPU Runs for experimentation 

## TPU Run 1 : Stability Experiments

**Date:** May 3, 2026  
**Hardware:** Kaggle TPU v5e-8  
**Model:** 1.3M parameter hyperbolic coding model  
**Data:** bigcode/the-stack, Python subset, streaming  

**Objective:**  
Run 5 experiments to find the stable training configuration
for hyperbolic attention. Each experiment isolates one variable
to identify which combination of normalization strategies,
distance approximations, and optimizers produces stable training.

**Experiments:**  
- A: Euclidean baseline (control)  
- B: Hyperbolic, no stabilization  
- C: Hyperbolic, norm clipping (max=8.0)  
- D: Hyperbolic, norm clipping + learnable temperature  
- E: Hyperbolic, norm clipping + learnable temperature + RAdam  

**Success criteria:**  
Loss decreases without NaN across 1000 steps.  
Embedding norms stay bounded.  
Geometry-AST correlation is preserved after training.

In [1]:
import jax
print(jax.devices())

/usr/local/lib/python3.12/site-packages/jax/_src/cloud_tpu_init.py:93: UserWarning: Transparent hugepages are not enabled. TPU runtime startup and shutdown time should be significantly improved on TPU v5e and newer. If not already set, you may need to enable transparent hugepages in your VM image (sudo sh -c "echo always > /sys/kernel/mm/transparent_hugepage/enabled")
  warnings.warn(
E0000 00:00:1777846747.861757      14 common_lib.cc:648] Could not set metric server port: INVALID_ARGUMENT: Could not find SliceBuilder port 8471 in any of the 0 ports provided in `tpu_process_addresses`="local"
=== Source Location Trace: === 
learning/45eac/tfrc/runtime/common_lib.cc:238


[TpuDevice(id=0, process_index=0, coords=(0,0,0), core_on_chip=0), TpuDevice(id=1, process_index=0, coords=(1,0,0), core_on_chip=0), TpuDevice(id=2, process_index=0, coords=(0,1,0), core_on_chip=0), TpuDevice(id=3, process_index=0, coords=(1,1,0), core_on_chip=0), TpuDevice(id=4, process_index=0, coords=(0,2,0), core_on_chip=0), TpuDevice(id=5, process_index=0, coords=(1,2,0), core_on_chip=0), TpuDevice(id=6, process_index=0, coords=(0,3,0), core_on_chip=0), TpuDevice(id=7, process_index=0, coords=(1,3,0), core_on_chip=0)]


### Load data for further experiments 

In [26]:
import numpy as np
import os

# Settings
# so we never have to redownload
SEQ_LEN    = 2048
N_SEQS     = 1000  # one sequence per file, combine into batches at load time
SAVE_DIR   = "/kaggle/working/batches"

os.makedirs(SAVE_DIR, exist_ok=True)

def make_single_sequence(dataset_iter, seq_len=2048):
    """Pull one sequence from the stream"""
    while True:
        try:
            example = next(dataset_iter)
        except StopIteration:
            return None

        source = example["content"]
        if len(source) > 50_000:
            continue

        tree = parse_code(source)
        if tree is None:
            continue

        nodes, edges = extract_ast_edges(tree)
        token_ids, _, depths = align_tokens_to_ast(source, nodes)

        # Truncate to seq_len
        token_ids = token_ids[:seq_len]
        depths    = depths[:seq_len]

        # Pad if shorter
        pad_len   = seq_len - len(token_ids)
        token_ids = token_ids + [0] * pad_len
        depths    = depths    + [0] * pad_len

        # Clip vocab
        token_ids = [min(t, 49151) for t in token_ids]

        return {
            "token_ids": np.array(token_ids, dtype=np.int32),
            "depths":    np.array(depths,    dtype=np.int32),
        }

# Prefetch and save
dataset_iter = iter(dataset)
saved        = 0

print(f"Saving {N_SEQS} sequences at seq_len={SEQ_LEN}")
print()

while saved < N_SEQS:
    seq = make_single_sequence(dataset_iter, seq_len=SEQ_LEN)
    if seq is None:
        print("Dataset exhausted")
        break

    np.savez(
        f"{SAVE_DIR}/seq_{saved:04d}.npz",
        token_ids = seq["token_ids"],
        depths    = seq["depths"]
    )

    saved += 1
    if saved % 100 == 0:
        size = sum(os.path.getsize(f"{SAVE_DIR}/{f}")
                   for f in os.listdir(SAVE_DIR))
        print(f"  Saved: {saved}/{N_SEQS} "
              f"| Disk: {size/1e6:.1f}MB")

print(f"\nDone {saved} sequences saved")
total = sum(os.path.getsize(f"{SAVE_DIR}/{f}")
            for f in os.listdir(SAVE_DIR))
print(f"Total size: {total/1e6:.1f}MB")

Saving 1000 sequences at seq_len=2048
This takes ~20-30 minutes on CPU

  Saved: 100/1000 | Disk: 1.7MB
  Saved: 200/1000 | Disk: 3.4MB
  Saved: 300/1000 | Disk: 5.1MB
  Saved: 400/1000 | Disk: 6.8MB
  Saved: 500/1000 | Disk: 8.4MB
  Saved: 600/1000 | Disk: 10.1MB
  Saved: 700/1000 | Disk: 11.8MB
  Saved: 800/1000 | Disk: 13.5MB
  Saved: 900/1000 | Disk: 15.2MB
  Saved: 1000/1000 | Disk: 16.9MB

Done — 1000 sequences saved
Total size: 16.9MB


### Save data for loading into TPUs

In [59]:
# Quick check before publishing
test = np.load(f"{SAVE_DIR}/seq_0000.npz")
print(f"token_ids shape: {test['token_ids'].shape}")
print(f"depths shape:    {test['depths'].shape}")
print(f"token_ids range: {test['token_ids'].min()} to {test['token_ids'].max()}")
print(f"depths range:    {test['depths'].min()} to {test['depths'].max()}")

# All should pass
assert test['token_ids'].shape == (2048,)
assert test['depths'].shape    == (2048,)
assert test['token_ids'].max() <= 49151
print("\n Data looks correct")

token_ids shape: (2048,)
depths shape:    (2048,)
token_ids range: 0 to 43499
depths range:    0 to 14

 Data looks correct


# Experiment A 

In [49]:
N_DEVICES = len(jax.devices())
print(f"Number of devices: {N_DEVICES}")  # should print 8

def load_batch(step, batch_per_device=8, seq_len=2048):
    total_seqs = N_DEVICES * batch_per_device

    token_ids_list = []
    depths_list    = []

    for i in range(total_seqs):
        idx  = (step * total_seqs + i) % 1000
        data = np.load(f"{DATA_DIR}/seq_{idx:04d}.npz")
        token_ids_list.append(data["token_ids"][:seq_len])
        depths_list.append(data["depths"][:seq_len])

    token_ids = np.array(token_ids_list, dtype=np.int32).reshape(
        N_DEVICES, batch_per_device, seq_len)
    depths    = np.array(depths_list,    dtype=np.int32).reshape(
        N_DEVICES, batch_per_device, seq_len)

    return {
        "token_ids": jnp.array(token_ids),
        "depths":    jnp.array(depths),
    }

# Test
batch = load_batch(0)
print("token_ids shape:", batch["token_ids"].shape)
# Should print: (8, 8, 2048)

Number of devices: 8
token_ids shape: (8, 8, 2048)


In [51]:
class EuclideanAttention(nn.Module):
    num_heads: int
    head_dim:  int

    @nn.compact
    def __call__(self, x):
        B, S, D = x.shape
        H, Hd   = self.num_heads, self.head_dim

        Q = nn.Dense(H * Hd)(x).reshape(B, S, H, Hd).transpose(0,2,1,3)
        K = nn.Dense(H * Hd)(x).reshape(B, S, H, Hd).transpose(0,2,1,3)
        V = nn.Dense(H * Hd)(x).reshape(B, S, H, Hd).transpose(0,2,1,3)

        scores  = jnp.einsum("bhid,bhjd->bhij", Q, K)
        scores  = scores / jnp.sqrt(jnp.array(Hd, dtype=jnp.float32))
        scores  = scores - jnp.max(scores, axis=-1, keepdims=True)
        weights = jax.nn.softmax(scores, axis=-1)

        out = jnp.einsum("bhij,bhjd->bhid", weights, V)
        out = out.transpose(0, 2, 1, 3).reshape(B, S, H * Hd)
        return nn.Dense(D)(out)


class EuclideanCodingModel(nn.Module):
    vocab_size: int = 49152
    hidden_dim: int = 16
    num_layers: int = 4
    num_heads:  int = 4
    max_depth:  int = 20

    @nn.compact
    def __call__(self, token_ids, depths):
        tok_emb = nn.Embed(self.vocab_size,
                            self.hidden_dim)(token_ids)
        dep_emb = nn.Embed(self.max_depth,
                            self.hidden_dim)(
                                jnp.clip(depths, 0, self.max_depth - 1))
        x = tok_emb + dep_emb

        for _ in range(self.num_layers):
            residual = x
            attn_out = EuclideanAttention(
                self.num_heads,
                self.hidden_dim // self.num_heads
            )(x)
            x = nn.LayerNorm()(residual + attn_out)

            ffn = nn.Dense(self.hidden_dim * 4)(x)
            ffn = nn.gelu(ffn)
            ffn = nn.Dense(self.hidden_dim)(ffn)
            x   = nn.LayerNorm()(x + ffn)

        return nn.Dense(self.vocab_size)(x)

# Initialize model
model        = EuclideanCodingModel()
key          = jax.random.PRNGKey(0)
dummy_tokens = jnp.ones((2, 64), dtype=jnp.int32)
dummy_depths = jnp.ones((2, 64), dtype=jnp.int32)
params       = model.init(key, dummy_tokens, dummy_depths)

n_params = sum(x.size for x in
               jax.tree_util.tree_leaves(params))
print(f"Parameters: {n_params/1e6:.2f}M")

# Initialize optimizer
optimizer = optax.adam(1e-3)
opt_state = optimizer.init(params)

# Replicate across all 8 TPUs
params_replicated    = jax.device_put_replicated(
    params,    jax.devices())
opt_state_replicated = jax.device_put_replicated(
    opt_state, jax.devices())

print(f"Replicated across {N_DEVICES} devices")

Parameters: 1.64M
Replicated across 8 devices


In [53]:
from functools import partial
import json

@partial(jax.pmap, axis_name="devices")
def train_step(params, opt_state, batch):
    def loss_fn(params):
        logits  = model.apply(params,
                               batch["token_ids"],
                               batch["depths"])
        targets = batch["token_ids"][:, 1:]
        logits  = logits[:, :-1, :]
        return optax.softmax_cross_entropy_with_integer_labels(
            logits, targets).mean()

    loss, grads = jax.value_and_grad(loss_fn)(params)

    # Average gradients across all 8 TPUs
    grads = jax.lax.pmean(grads, axis_name="devices")
    loss  = jax.lax.pmean(loss,  axis_name="devices")

    updates, new_opt_state = optimizer.update(grads, opt_state)
    new_params = optax.apply_updates(params, updates)
    return loss, new_params, new_opt_state

print("=" * 50)
print("Experiment A : Euclidean Baseline")
print("=" * 50)

results_A = []
n_steps   = 1000

for step in range(n_steps):
    batch = load_batch(step, batch_per_device=8, seq_len=2048)

    loss, params_replicated, opt_state_replicated = train_step(
        params_replicated,
        opt_state_replicated,
        batch
    )

    loss_scalar = float(loss[0])

    if jnp.isnan(loss[0]):
        print(f"NaN at step {step}")
        break

    results_A.append(loss_scalar)

    if step % 100 == 0:
        print(f"Step {step:4d} | Loss: {loss_scalar:.4f}")

print(f"\nExperiment A done")
print(f"Final loss (avg last 50): {np.mean(results_A[-50:]):.4f}")

with open("/kaggle/working/results_A.json", "w") as f:
    json.dump(results_A, f)
print("Saved")

Experiment A : Euclidean Baseline
Step    0 | Loss: 12.1337
Step  100 | Loss: 4.4194
Step  200 | Loss: 1.6513
Step  300 | Loss: 1.3691
Step  400 | Loss: 1.3952
Step  500 | Loss: 1.2291
Step  600 | Loss: 1.1460
Step  700 | Loss: 1.0592
Step  800 | Loss: 1.0050
Step  900 | Loss: 1.0773

Experiment A done
Final loss (avg last 50): 0.9985
Saved


# Experiment A : Euclidean Baseline Result

**Date:** May 3, 2026  
**Configuration:**
- Model: EuclideanCodingModel
- Parameters: 1.64M
- Optimizer: Adam (lr=1e-3)
- Devices: 8 × TPU v5e
- Batch per device: 8
- Seq len: 2048
- Total tokens per step: 131,072
- Total tokens seen: ~131M
- Steps: 1000

**Result:**

| Step | Loss |
|------|------|
| 0    | 12.1337 |
| 100  | 4.4194  |
| 300  | 1.3691  |
| 500  | 1.2291  |
| 700  | 1.0592  |
| 1000 | 0.9985  |

**Final loss (avg last 50 steps): 0.9985**

The Euclidean baseline trained stably across 1000 steps with no NaN. Loss dropped from 12.13 at random initialization to 0.99 at convergence, showing consistent improvement throughout. This establishes the control result that all hyperbolic experiments will be compared against. Any hyperbolic model that converges to a lower final loss on the same data has demonstrated a geometric advantage.

# Experiment B

In [60]:
class HyperbolicAttention(nn.Module):
    num_heads: int
    head_dim:  int

    @nn.compact
    def __call__(self, x):
        B, S, D = x.shape
        H, Hd   = self.num_heads, self.head_dim

        Q = nn.Dense(H * Hd)(x).reshape(B, S, H, Hd).transpose(0,2,1,3)
        K = nn.Dense(H * Hd)(x).reshape(B, S, H, Hd).transpose(0,2,1,3)
        V = nn.Dense(H * Hd)(x).reshape(B, S, H, Hd).transpose(0,2,1,3)

        def to_hyperboloid(z):
            z  = jnp.clip(z, -8.0, 8.0)
            z0 = jnp.sqrt(1 + jnp.sum(z**2, axis=-1, keepdims=True))
            return jnp.concatenate([z0, z], axis=-1)

        Q_h = to_hyperboloid(Q)
        K_h = to_hyperboloid(K)

        Q_s = Q_h[..., 1:] / jnp.maximum(
            jnp.linalg.norm(Q_h[..., 1:], axis=-1, keepdims=True), 1e-8)
        K_s = K_h[..., 1:] / jnp.maximum(
            jnp.linalg.norm(K_h[..., 1:], axis=-1, keepdims=True), 1e-8)

        cos_sim    = jnp.einsum("bhid,bhjd->bhij", Q_s, K_s)
        scale_term = (Q_h[..., 0:1] *
                      K_h[..., 0:1].transpose(0, 1, 3, 2))
        scores     = -scale_term * (1 - cos_sim)

        log_t   = self.param('log_temp', nn.initializers.zeros, ())
        temp    = jnp.maximum(jnp.exp(log_t), 0.1)
        scores  = scores / temp
        scores  = scores - jnp.max(scores, axis=-1, keepdims=True)
        weights = jax.nn.softmax(scores, axis=-1)

        out = jnp.einsum("bhij,bhjd->bhid", weights, V)
        out = out.transpose(0, 2, 1, 3).reshape(B, S, H * Hd)
        return nn.Dense(D)(out)


class HyperbolicCodingModel(nn.Module):
    vocab_size: int = 49152
    hidden_dim: int = 16
    num_layers: int = 4
    num_heads:  int = 4
    max_depth:  int = 20

    @nn.compact
    def __call__(self, token_ids, depths):
        tok_emb = nn.Embed(self.vocab_size,
                            self.hidden_dim)(token_ids)
        dep_emb = nn.Embed(self.max_depth,
                            self.hidden_dim)(
                                jnp.clip(depths, 0, self.max_depth - 1))
        x = tok_emb + dep_emb

        spatial = x / jnp.maximum(
            jnp.linalg.norm(x, axis=-1, keepdims=True), 1e-8)
        depth_f = depths.astype(jnp.float32)
        scale   = jnp.exp(depth_f * 0.1)[..., None]
        spatial = jnp.clip(spatial * scale, -8.0, 8.0)

        for _ in range(self.num_layers):
            residual = spatial
            attn_out = HyperbolicAttention(
                self.num_heads,
                self.hidden_dim // self.num_heads
            )(spatial)
            spatial = nn.LayerNorm()(residual + attn_out)
            spatial = jnp.clip(spatial, -8.0, 8.0)

            ffn     = nn.Dense(self.hidden_dim * 4)(spatial)
            ffn     = nn.gelu(ffn)
            ffn     = nn.Dense(self.hidden_dim)(ffn)
            spatial = nn.LayerNorm()(spatial + ffn)
            spatial = jnp.clip(spatial, -8.0, 8.0)

        return nn.Dense(self.vocab_size)(spatial)

#Initialize
model        = HyperbolicCodingModel()
key          = jax.random.PRNGKey(0)
dummy_tokens = jnp.ones((2, 64), dtype=jnp.int32)
dummy_depths = jnp.ones((2, 64), dtype=jnp.int32)
params       = model.init(key, dummy_tokens, dummy_depths)

n_params = sum(x.size for x in
               jax.tree_util.tree_leaves(params))
print(f"Parameters: {n_params/1e6:.2f}M")

optimizer    = optax.adam(1e-3)
opt_state    = optimizer.init(params)

params_replicated    = jax.device_put_replicated(
    params,    jax.devices())
opt_state_replicated = jax.device_put_replicated(
    opt_state, jax.devices())

print(f"Replicated across {N_DEVICES} devices")

Parameters: 1.64M
Replicated across 8 devices


In [61]:
import threading
from queue import Queue
from functools import partial

# Threaded prefetch
def prefetch_worker(queue, n_steps,
                    batch_per_device=8, seq_len=2048):
    for step in range(n_steps):
        batch = load_batch(step, batch_per_device, seq_len)
        queue.put(batch)
    queue.put(None)

# pmap train step
@partial(jax.pmap, axis_name="devices")
def train_step_B(params, opt_state, batch):
    def loss_fn(params):
        logits  = model.apply(params,
                               batch["token_ids"],
                               batch["depths"])
        targets = batch["token_ids"][:, 1:]
        logits  = logits[:, :-1, :]
        return optax.softmax_cross_entropy_with_integer_labels(
            logits, targets).mean()

    loss, grads = jax.value_and_grad(loss_fn)(params)
    grads = jax.lax.pmean(grads, axis_name="devices")
    loss  = jax.lax.pmean(loss,  axis_name="devices")

    updates, new_opt_state = optimizer.update(grads, opt_state)
    new_params = optax.apply_updates(params, updates)
    return loss, new_params, new_opt_state

# Start prefetch thread
n_steps      = 1000
batch_queue  = Queue(maxsize=8)
thread       = threading.Thread(
    target=prefetch_worker,
    args=(batch_queue, n_steps),
    kwargs={"batch_per_device": 8, "seq_len": 2048},
    daemon=True
)
thread.start()

print("=" * 50)
print("Experiment B : Hyperbolic, No Stabilization")
print("=" * 50)

results_B = []

for step in range(n_steps):
    batch = batch_queue.get()
    if batch is None:
        break

    loss, params_replicated, opt_state_replicated = train_step_B(
        params_replicated,
        opt_state_replicated,
        batch
    )

    loss_scalar = float(loss[0])

    if jnp.isnan(loss[0]):
        print(f"NaN at step {step}")
        break

    results_B.append(loss_scalar)

    if step % 100 == 0:
        print(f"Step {step:4d} | Loss: {loss_scalar:.4f}")

print(f"\nExperiment B done")
print(f"Final loss (avg last 50): {np.mean(results_B[-50:]):.4f}")

with open("/kaggle/working/results_B.json", "w") as f:
    json.dump(results_B, f)
print("Saved")

Experiment B : Hyperbolic, No Stabilization
Step    0 | Loss: 11.6722
Step  100 | Loss: 4.3709
Step  200 | Loss: 1.6535
Step  300 | Loss: 1.3764
Step  400 | Loss: 1.4249
Step  500 | Loss: 1.2910
Step  600 | Loss: 1.2265
Step  700 | Loss: 1.1363
Step  800 | Loss: 1.0816
Step  900 | Loss: 1.1443

Experiment B done
Final loss (avg last 50): 1.0583
Saved


# Experiment B : Hyperbolic, No Stabilization Result

**Date:** May 3, 2026  
**Configuration:**
- Model: HyperbolicCodingModel
- Parameters: 1.64M
- Optimizer: Adam (lr=1e-3)
- Devices: 8 x TPU v5e
- Batch per device: 8
- Seq len: 2048
- Total tokens per step: 131,072
- Total tokens seen: ~131M
- Steps: 1000

**Result:**

| Step | Loss |
|------|------|
| 0    | 11.6722 |
| 100  | 4.3709  |
| 300  | 1.3764  |
| 500  | 1.2910  |
| 700  | 1.1363  |
| 1000 | 1.0583  |

**Final loss (avg last 50 steps): 1.0583**

The hyperbolic model trained stably across 1000 steps with no NaN
and no additional stabilization techniques. Loss dropped from
11.67 at random initialization to 1.06 at convergence, showing
consistent improvement throughout with no signs of gradient
explosion or numerical instability.

# Comparison : Experiment A vs Experiment B

**Date:** May 3, 2026

## Overview

| Metric | Experiment A (Euclidean) | Experiment B (Hyperbolic) |
|--------|--------------------------|---------------------------|
| Model | EuclideanCodingModel | HyperbolicCodingModel |
| Parameters | 1.64M | 1.64M |
| Optimizer | Adam (lr=1e-3) | Adam (lr=1e-3) |
| Steps | 1000 | 1000 |
| Seq len | 2048 | 2048 |
| Batch per device | 8 | 8 |
| Devices | 8 x TPU v5e | 8 x TPU v5e |

## Loss Curves

| Step | Euclidean | Hyperbolic |
|------|-----------|------------|
| 0    | 12.1337   | 11.6722    |
| 100  | 4.4194    | 4.3709     |
| 300  | 1.3691    | 1.3764     |
| 500  | 1.2291    | 1.2910     |
| 700  | 1.0592    | 1.1363     |
| 900  | 1.0773    | 1.1443     |
| Final loss (avg last 50) | **0.9985** | **1.0583** |

## Observations

Both models trained stably with no NaN across 1000 steps. The
Euclidean baseline converged to a lower final loss of 0.9985
compared to the hyperbolic model at 1.0583, a difference of
0.0598. Both models showed similar learning dynamics in the
early steps, with the Euclidean model pulling ahead after
step 300 and maintaining that advantage through convergence.

## Interpretation

The hyperbolic model did not outperform the Euclidean baseline
at this scale under standard Adam optimization. This is expected
given that standard Adam does not respect the Riemannian geometry
of the hyperboloid, meaning parameter updates can push points
off the manifold and degrade the geometric structure that the
model is trying to exploit. Experiments C, D, and E will
isolate the effect of norm clipping, learnable temperature,
and Riemannian Adam respectively to determine which
stabilization technique closes the gap.

## Key Finding

Hyperbolic training is stable without any additional
stabilization techniques. The geometry does not cause
divergence or NaN under standard training. The question
for remaining experiments is whether targeted stabilization
can unlock the theoretical geometric advantage.

In [63]:
import gc
import jax

try:
    del params_replicated, opt_state_replicated
    del params, opt_state
except:
    pass

gc.collect()
jax.clear_caches()

print("Cleared")

Cleared


# Experiment A Phase 2

In [64]:
class EuclideanAttention(nn.Module):
    num_heads: int
    head_dim:  int

    @nn.compact
    def __call__(self, x):
        B, S, D = x.shape
        H, Hd   = self.num_heads, self.head_dim

        Q = nn.Dense(H * Hd)(x).reshape(B, S, H, Hd).transpose(0,2,1,3)
        K = nn.Dense(H * Hd)(x).reshape(B, S, H, Hd).transpose(0,2,1,3)
        V = nn.Dense(H * Hd)(x).reshape(B, S, H, Hd).transpose(0,2,1,3)

        scores  = jnp.einsum("bhid,bhjd->bhij", Q, K)
        scores  = scores / jnp.sqrt(jnp.array(Hd, dtype=jnp.float32))
        scores  = scores - jnp.max(scores, axis=-1, keepdims=True)
        weights = jax.nn.softmax(scores, axis=-1)

        out = jnp.einsum("bhij,bhjd->bhid", weights, V)
        out = out.transpose(0, 2, 1, 3).reshape(B, S, H * Hd)
        return nn.Dense(D)(out)


class EuclideanCodingModel(nn.Module):
    vocab_size: int = 49152
    hidden_dim: int = 16
    num_layers: int = 4
    num_heads:  int = 4
    max_depth:  int = 20

    @nn.compact
    def __call__(self, token_ids, depths):
        tok_emb = nn.Embed(self.vocab_size,
                            self.hidden_dim)(token_ids)
        dep_emb = nn.Embed(self.max_depth,
                            self.hidden_dim)(
                                jnp.clip(depths, 0, self.max_depth - 1))
        x = tok_emb + dep_emb

        for _ in range(self.num_layers):
            residual = x
            attn_out = EuclideanAttention(
                self.num_heads,
                self.hidden_dim // self.num_heads
            )(x)
            x = nn.LayerNorm()(residual + attn_out)

            ffn = nn.Dense(self.hidden_dim * 4)(x)
            ffn = nn.gelu(ffn)
            ffn = nn.Dense(self.hidden_dim)(ffn)
            x   = nn.LayerNorm()(x + ffn)

        return nn.Dense(self.vocab_size)(x)

# Initialize model
model        = EuclideanCodingModel()
key          = jax.random.PRNGKey(0)
dummy_tokens = jnp.ones((2, 64), dtype=jnp.int32)
dummy_depths = jnp.ones((2, 64), dtype=jnp.int32)
params       = model.init(key, dummy_tokens, dummy_depths)

n_params = sum(x.size for x in
               jax.tree_util.tree_leaves(params))
print(f"Parameters: {n_params/1e6:.2f}M")

# Initialize optimizer
optimizer = optax.adam(1e-3)
opt_state = optimizer.init(params)

# Replicate across all 8 TPUs
params_replicated    = jax.device_put_replicated(
    params,    jax.devices())
opt_state_replicated = jax.device_put_replicated(
    opt_state, jax.devices())

print(f"Replicated across {N_DEVICES} devices")

Parameters: 1.64M
Replicated across 8 devices


In [66]:
from functools import partial
import json

@partial(jax.pmap, axis_name="devices")
def train_step(params, opt_state, batch):
    def loss_fn(params):
        logits  = model.apply(params,
                               batch["token_ids"],
                               batch["depths"])
        targets = batch["token_ids"][:, 1:]
        logits  = logits[:, :-1, :]
        return optax.softmax_cross_entropy_with_integer_labels(
            logits, targets).mean()

    loss, grads = jax.value_and_grad(loss_fn)(params)

    # Average gradients across all 8 TPUs
    grads = jax.lax.pmean(grads, axis_name="devices")
    loss  = jax.lax.pmean(loss,  axis_name="devices")

    updates, new_opt_state = optimizer.update(grads, opt_state)
    new_params = optax.apply_updates(params, updates)
    return loss, new_params, new_opt_state

print("=" * 50)
print("Experiment A : Euclidean Baseline Phase 2 (3000 steps)")
print("=" * 50)

results_A = []
n_steps   = 3000

for step in range(n_steps):
    batch = load_batch(step, batch_per_device=8, seq_len=2048)

    loss, params_replicated, opt_state_replicated = train_step(
        params_replicated,
        opt_state_replicated,
        batch
    )

    loss_scalar = float(loss[0])

    if jnp.isnan(loss[0]):
        print(f"NaN at step {step}")
        break

    results_A.append(loss_scalar)

    if step % 100 == 0:
        print(f"Step {step:4d} | Loss: {loss_scalar:.4f}")

print(f"\nExperiment A done")
print(f"Final loss (avg last 50): {np.mean(results_A[-50:]):.4f}")

with open("/kaggle/working/results_A.json", "w") as f:
    json.dump(results_A, f)
print("Saved")

Experiment A : Euclidean Baseline Phase 2 (3000 steps)
Step    0 | Loss: 12.1337
Step  100 | Loss: 4.4194
Step  200 | Loss: 1.6513
Step  300 | Loss: 1.3691
Step  400 | Loss: 1.3952
Step  500 | Loss: 1.2291
Step  600 | Loss: 1.1460
Step  700 | Loss: 1.0592
Step  800 | Loss: 1.0050
Step  900 | Loss: 1.0773
Step 1000 | Loss: 0.9797
Step 1100 | Loss: 0.9453
Step 1200 | Loss: 0.9136
Step 1300 | Loss: 0.8789
Step 1400 | Loss: 0.9628
Step 1500 | Loss: 0.8865
Step 1600 | Loss: 0.8628
Step 1700 | Loss: 0.8482
Step 1800 | Loss: 0.8155
Step 1900 | Loss: 0.8900
Step 2000 | Loss: 0.8305
Step 2100 | Loss: 0.8090
Step 2200 | Loss: 0.7917
Step 2300 | Loss: 0.7672
Step 2400 | Loss: 0.8419
Step 2500 | Loss: 0.7879
Step 2600 | Loss: 0.7683
Step 2700 | Loss: 0.7626
Step 2800 | Loss: 0.7326
Step 2900 | Loss: 0.8337

Experiment A done
Final loss (avg last 50): 0.7628
Saved


# Experiment B Phase 2

In [67]:
class HyperbolicAttention(nn.Module):
    num_heads: int
    head_dim:  int

    @nn.compact
    def __call__(self, x):
        B, S, D = x.shape
        H, Hd   = self.num_heads, self.head_dim

        Q = nn.Dense(H * Hd)(x).reshape(B, S, H, Hd).transpose(0,2,1,3)
        K = nn.Dense(H * Hd)(x).reshape(B, S, H, Hd).transpose(0,2,1,3)
        V = nn.Dense(H * Hd)(x).reshape(B, S, H, Hd).transpose(0,2,1,3)

        def to_hyperboloid(z):
            z  = jnp.clip(z, -8.0, 8.0)
            z0 = jnp.sqrt(1 + jnp.sum(z**2, axis=-1, keepdims=True))
            return jnp.concatenate([z0, z], axis=-1)

        Q_h = to_hyperboloid(Q)
        K_h = to_hyperboloid(K)

        Q_s = Q_h[..., 1:] / jnp.maximum(
            jnp.linalg.norm(Q_h[..., 1:], axis=-1, keepdims=True), 1e-8)
        K_s = K_h[..., 1:] / jnp.maximum(
            jnp.linalg.norm(K_h[..., 1:], axis=-1, keepdims=True), 1e-8)

        cos_sim    = jnp.einsum("bhid,bhjd->bhij", Q_s, K_s)
        scale_term = (Q_h[..., 0:1] *
                      K_h[..., 0:1].transpose(0, 1, 3, 2))
        scores     = -scale_term * (1 - cos_sim)

        log_t   = self.param('log_temp', nn.initializers.zeros, ())
        temp    = jnp.maximum(jnp.exp(log_t), 0.1)
        scores  = scores / temp
        scores  = scores - jnp.max(scores, axis=-1, keepdims=True)
        weights = jax.nn.softmax(scores, axis=-1)

        out = jnp.einsum("bhij,bhjd->bhid", weights, V)
        out = out.transpose(0, 2, 1, 3).reshape(B, S, H * Hd)
        return nn.Dense(D)(out)


class HyperbolicCodingModel(nn.Module):
    vocab_size: int = 49152
    hidden_dim: int = 16
    num_layers: int = 4
    num_heads:  int = 4
    max_depth:  int = 20

    @nn.compact
    def __call__(self, token_ids, depths):
        tok_emb = nn.Embed(self.vocab_size,
                            self.hidden_dim)(token_ids)
        dep_emb = nn.Embed(self.max_depth,
                            self.hidden_dim)(
                                jnp.clip(depths, 0, self.max_depth - 1))
        x = tok_emb + dep_emb

        spatial = x / jnp.maximum(
            jnp.linalg.norm(x, axis=-1, keepdims=True), 1e-8)
        depth_f = depths.astype(jnp.float32)
        scale   = jnp.exp(depth_f * 0.1)[..., None]
        spatial = jnp.clip(spatial * scale, -8.0, 8.0)

        for _ in range(self.num_layers):
            residual = spatial
            attn_out = HyperbolicAttention(
                self.num_heads,
                self.hidden_dim // self.num_heads
            )(spatial)
            spatial = nn.LayerNorm()(residual + attn_out)
            spatial = jnp.clip(spatial, -8.0, 8.0)

            ffn     = nn.Dense(self.hidden_dim * 4)(spatial)
            ffn     = nn.gelu(ffn)
            ffn     = nn.Dense(self.hidden_dim)(ffn)
            spatial = nn.LayerNorm()(spatial + ffn)
            spatial = jnp.clip(spatial, -8.0, 8.0)

        return nn.Dense(self.vocab_size)(spatial)

#Initialize
model        = HyperbolicCodingModel()
key          = jax.random.PRNGKey(0)
dummy_tokens = jnp.ones((2, 64), dtype=jnp.int32)
dummy_depths = jnp.ones((2, 64), dtype=jnp.int32)
params       = model.init(key, dummy_tokens, dummy_depths)

n_params = sum(x.size for x in
               jax.tree_util.tree_leaves(params))
print(f"Parameters: {n_params/1e6:.2f}M")

optimizer    = optax.adam(1e-3)
opt_state    = optimizer.init(params)

params_replicated    = jax.device_put_replicated(
    params,    jax.devices())
opt_state_replicated = jax.device_put_replicated(
    opt_state, jax.devices())

print(f"Replicated across {N_DEVICES} devices")

Parameters: 1.64M
Replicated across 8 devices


In [68]:
import threading
from queue import Queue
from functools import partial

# Threaded prefetch
def prefetch_worker(queue, n_steps,
                    batch_per_device=8, seq_len=2048):
    for step in range(n_steps):
        batch = load_batch(step, batch_per_device, seq_len)
        queue.put(batch)
    queue.put(None)

# pmap train step
@partial(jax.pmap, axis_name="devices")
def train_step_B(params, opt_state, batch):
    def loss_fn(params):
        logits  = model.apply(params,
                               batch["token_ids"],
                               batch["depths"])
        targets = batch["token_ids"][:, 1:]
        logits  = logits[:, :-1, :]
        return optax.softmax_cross_entropy_with_integer_labels(
            logits, targets).mean()

    loss, grads = jax.value_and_grad(loss_fn)(params)
    grads = jax.lax.pmean(grads, axis_name="devices")
    loss  = jax.lax.pmean(loss,  axis_name="devices")

    updates, new_opt_state = optimizer.update(grads, opt_state)
    new_params = optax.apply_updates(params, updates)
    return loss, new_params, new_opt_state

# Start prefetch thread
n_steps      = 3000
batch_queue  = Queue(maxsize=8)
thread       = threading.Thread(
    target=prefetch_worker,
    args=(batch_queue, n_steps),
    kwargs={"batch_per_device": 8, "seq_len": 2048},
    daemon=True
)
thread.start()

print("=" * 50)
print("Experiment B : Hyperbolic, No Stabilization Phase 2 (3000 steps) ")
print("=" * 50)

results_B = []

for step in range(n_steps):
    batch = batch_queue.get()
    if batch is None:
        break

    loss, params_replicated, opt_state_replicated = train_step_B(
        params_replicated,
        opt_state_replicated,
        batch
    )

    loss_scalar = float(loss[0])

    if jnp.isnan(loss[0]):
        print(f"NaN at step {step}")
        break

    results_B.append(loss_scalar)

    if step % 100 == 0:
        print(f"Step {step:4d} | Loss: {loss_scalar:.4f}")

print(f"\nExperiment B done")
print(f"Final loss (avg last 50): {np.mean(results_B[-50:]):.4f}")

with open("/kaggle/working/results_B_2.json", "w") as f:
    json.dump(results_B, f)
print("Saved")

Experiment B : Hyperbolic, No Stabilization Phase 2 (3000 steps) 
Step    0 | Loss: 11.6722
Step  100 | Loss: 4.3709
Step  200 | Loss: 1.6535
Step  300 | Loss: 1.3764
Step  400 | Loss: 1.4249
Step  500 | Loss: 1.2910
Step  600 | Loss: 1.2265
Step  700 | Loss: 1.1363
Step  800 | Loss: 1.0816
Step  900 | Loss: 1.1443
Step 1000 | Loss: 1.0371
Step 1100 | Loss: 0.9931
Step 1200 | Loss: 0.9535
Step 1300 | Loss: 0.9155
Step 1400 | Loss: 0.9905
Step 1500 | Loss: 0.9235
Step 1600 | Loss: 0.8876
Step 1700 | Loss: 0.8701
Step 1800 | Loss: 0.8314
Step 1900 | Loss: 0.9131
Step 2000 | Loss: 0.8529
Step 2100 | Loss: 0.8254
Step 2200 | Loss: 0.8153
Step 2300 | Loss: 0.7849
Step 2400 | Loss: 0.8648
Step 2500 | Loss: 0.8125
Step 2600 | Loss: 0.7809
Step 2700 | Loss: 0.7730
Step 2800 | Loss: 0.7483
Step 2900 | Loss: 0.8232

Experiment B done
Final loss (avg last 50): 0.7932
Saved


In [72]:
import gc
import jax

try:
    del params_replicated, opt_state_replicated
    del params, opt_state
except:
    pass

gc.collect()
jax.clear_caches()

print("Cleared")

Cleared


# Experiment A Phase 3

In [70]:
class EuclideanAttention(nn.Module):
    num_heads: int
    head_dim:  int

    @nn.compact
    def __call__(self, x):
        B, S, D = x.shape
        H, Hd   = self.num_heads, self.head_dim

        Q = nn.Dense(H * Hd)(x).reshape(B, S, H, Hd).transpose(0,2,1,3)
        K = nn.Dense(H * Hd)(x).reshape(B, S, H, Hd).transpose(0,2,1,3)
        V = nn.Dense(H * Hd)(x).reshape(B, S, H, Hd).transpose(0,2,1,3)

        scores  = jnp.einsum("bhid,bhjd->bhij", Q, K)
        scores  = scores / jnp.sqrt(jnp.array(Hd, dtype=jnp.float32))
        scores  = scores - jnp.max(scores, axis=-1, keepdims=True)
        weights = jax.nn.softmax(scores, axis=-1)

        out = jnp.einsum("bhij,bhjd->bhid", weights, V)
        out = out.transpose(0, 2, 1, 3).reshape(B, S, H * Hd)
        return nn.Dense(D)(out)


class EuclideanCodingModel(nn.Module):
    vocab_size: int = 49152
    hidden_dim: int = 16
    num_layers: int = 4
    num_heads:  int = 4
    max_depth:  int = 20

    @nn.compact
    def __call__(self, token_ids, depths):
        tok_emb = nn.Embed(self.vocab_size,
                            self.hidden_dim)(token_ids)
        dep_emb = nn.Embed(self.max_depth,
                            self.hidden_dim)(
                                jnp.clip(depths, 0, self.max_depth - 1))
        x = tok_emb + dep_emb

        for _ in range(self.num_layers):
            residual = x
            attn_out = EuclideanAttention(
                self.num_heads,
                self.hidden_dim // self.num_heads
            )(x)
            x = nn.LayerNorm()(residual + attn_out)

            ffn = nn.Dense(self.hidden_dim * 4)(x)
            ffn = nn.gelu(ffn)
            ffn = nn.Dense(self.hidden_dim)(ffn)
            x   = nn.LayerNorm()(x + ffn)

        return nn.Dense(self.vocab_size)(x)

# Initialize model
model        = EuclideanCodingModel()
key          = jax.random.PRNGKey(0)
dummy_tokens = jnp.ones((2, 64), dtype=jnp.int32)
dummy_depths = jnp.ones((2, 64), dtype=jnp.int32)
params       = model.init(key, dummy_tokens, dummy_depths)

n_params = sum(x.size for x in
               jax.tree_util.tree_leaves(params))
print(f"Parameters: {n_params/1e6:.2f}M")

# Initialize optimizer
optimizer = optax.adam(1e-3)
opt_state = optimizer.init(params)

# Replicate across all 8 TPUs
params_replicated    = jax.device_put_replicated(
    params,    jax.devices())
opt_state_replicated = jax.device_put_replicated(
    opt_state, jax.devices())

print(f"Replicated across {N_DEVICES} devices")

Parameters: 1.64M
Replicated across 8 devices


In [71]:
import threading
from queue import Queue
from functools import partial
import json

def prefetch_worker(queue, n_steps,
                    batch_per_device=8, seq_len=2048):
    for step in range(n_steps):
        batch = load_batch(step, batch_per_device, seq_len)
        queue.put(batch)
    queue.put(None)

@partial(jax.pmap, axis_name="devices")
def train_step_A(params, opt_state, batch):
    def loss_fn(params):
        logits  = model.apply(params,
                               batch["token_ids"],
                               batch["depths"])
        targets = batch["token_ids"][:, 1:]
        logits  = logits[:, :-1, :]
        return optax.softmax_cross_entropy_with_integer_labels(
            logits, targets).mean()

    loss, grads = jax.value_and_grad(loss_fn)(params)
    grads = jax.lax.pmean(grads, axis_name="devices")
    loss  = jax.lax.pmean(loss,  axis_name="devices")

    updates, new_opt_state = optimizer.update(grads, opt_state)
    new_params = optax.apply_updates(params, updates)
    return loss, new_params, new_opt_state

n_steps     = 5000
batch_queue = Queue(maxsize=8)
thread      = threading.Thread(
    target=prefetch_worker,
    args=(batch_queue, n_steps),
    kwargs={"batch_per_device": 8, "seq_len": 2048},
    daemon=True
)
thread.start()

print("=" * 50)
print("Experiment A : Euclidean Baseline Phase 3 (5000 steps)")
print("=" * 50)

results_A = []

for step in range(n_steps):
    batch = batch_queue.get()
    if batch is None:
        break

    loss, params_replicated, opt_state_replicated = train_step_A(
        params_replicated,
        opt_state_replicated,
        batch
    )

    loss_scalar = float(loss[0])

    if jnp.isnan(loss[0]):
        print(f"NaN at step {step}")
        break

    results_A.append(loss_scalar)

    if step % 100 == 0:
        print(f"Step {step:4d} | Loss: {loss_scalar:.4f}")

print(f"\nExperiment A done")
print(f"Final loss (avg last 50): {np.mean(results_A[-50:]):.4f}")

with open("/kaggle/working/results_A_3.json", "w") as f:
    json.dump(results_A, f)
print("Saved")

Experiment A : Euclidean Baseline Phase 3 (5000 steps)
Step    0 | Loss: 12.1337
Step  100 | Loss: 4.4194
Step  200 | Loss: 1.6513
Step  300 | Loss: 1.3691
Step  400 | Loss: 1.3952
Step  500 | Loss: 1.2291
Step  600 | Loss: 1.1460
Step  700 | Loss: 1.0592
Step  800 | Loss: 1.0050
Step  900 | Loss: 1.0773
Step 1000 | Loss: 0.9797
Step 1100 | Loss: 0.9453
Step 1200 | Loss: 0.9136
Step 1300 | Loss: 0.8789
Step 1400 | Loss: 0.9628
Step 1500 | Loss: 0.8865
Step 1600 | Loss: 0.8628
Step 1700 | Loss: 0.8482
Step 1800 | Loss: 0.8155
Step 1900 | Loss: 0.8900
Step 2000 | Loss: 0.8305
Step 2100 | Loss: 0.8090
Step 2200 | Loss: 0.7917
Step 2300 | Loss: 0.7672
Step 2400 | Loss: 0.8419
Step 2500 | Loss: 0.7879
Step 2600 | Loss: 0.7683
Step 2700 | Loss: 0.7626
Step 2800 | Loss: 0.7326
Step 2900 | Loss: 0.8337
Step 3000 | Loss: 0.7497
Step 3100 | Loss: 0.7310
Step 3200 | Loss: 0.7164
Step 3300 | Loss: 0.7067
Step 3400 | Loss: 0.7844
Step 3500 | Loss: 0.7276
Step 3600 | Loss: 0.7002
Step 3700 | Loss: 0

# Experiment B Phase 3

In [73]:
class HyperbolicAttention(nn.Module):
    num_heads: int
    head_dim:  int

    @nn.compact
    def __call__(self, x):
        B, S, D = x.shape
        H, Hd   = self.num_heads, self.head_dim

        Q = nn.Dense(H * Hd)(x).reshape(B, S, H, Hd).transpose(0,2,1,3)
        K = nn.Dense(H * Hd)(x).reshape(B, S, H, Hd).transpose(0,2,1,3)
        V = nn.Dense(H * Hd)(x).reshape(B, S, H, Hd).transpose(0,2,1,3)

        def to_hyperboloid(z):
            z  = jnp.clip(z, -8.0, 8.0)
            z0 = jnp.sqrt(1 + jnp.sum(z**2, axis=-1, keepdims=True))
            return jnp.concatenate([z0, z], axis=-1)

        Q_h = to_hyperboloid(Q)
        K_h = to_hyperboloid(K)

        Q_s = Q_h[..., 1:] / jnp.maximum(
            jnp.linalg.norm(Q_h[..., 1:], axis=-1, keepdims=True), 1e-8)
        K_s = K_h[..., 1:] / jnp.maximum(
            jnp.linalg.norm(K_h[..., 1:], axis=-1, keepdims=True), 1e-8)

        cos_sim    = jnp.einsum("bhid,bhjd->bhij", Q_s, K_s)
        scale_term = (Q_h[..., 0:1] *
                      K_h[..., 0:1].transpose(0, 1, 3, 2))
        scores     = -scale_term * (1 - cos_sim)

        log_t   = self.param('log_temp', nn.initializers.zeros, ())
        temp    = jnp.maximum(jnp.exp(log_t), 0.1)
        scores  = scores / temp
        scores  = scores - jnp.max(scores, axis=-1, keepdims=True)
        weights = jax.nn.softmax(scores, axis=-1)

        out = jnp.einsum("bhij,bhjd->bhid", weights, V)
        out = out.transpose(0, 2, 1, 3).reshape(B, S, H * Hd)
        return nn.Dense(D)(out)


class HyperbolicCodingModel(nn.Module):
    vocab_size: int = 49152
    hidden_dim: int = 16
    num_layers: int = 4
    num_heads:  int = 4
    max_depth:  int = 20

    @nn.compact
    def __call__(self, token_ids, depths):
        tok_emb = nn.Embed(self.vocab_size,
                            self.hidden_dim)(token_ids)
        dep_emb = nn.Embed(self.max_depth,
                            self.hidden_dim)(
                                jnp.clip(depths, 0, self.max_depth - 1))
        x = tok_emb + dep_emb

        spatial = x / jnp.maximum(
            jnp.linalg.norm(x, axis=-1, keepdims=True), 1e-8)
        depth_f = depths.astype(jnp.float32)
        scale   = jnp.exp(depth_f * 0.1)[..., None]
        spatial = jnp.clip(spatial * scale, -8.0, 8.0)

        for _ in range(self.num_layers):
            residual = spatial
            attn_out = HyperbolicAttention(
                self.num_heads,
                self.hidden_dim // self.num_heads
            )(spatial)
            spatial = nn.LayerNorm()(residual + attn_out)
            spatial = jnp.clip(spatial, -8.0, 8.0)

            ffn     = nn.Dense(self.hidden_dim * 4)(spatial)
            ffn     = nn.gelu(ffn)
            ffn     = nn.Dense(self.hidden_dim)(ffn)
            spatial = nn.LayerNorm()(spatial + ffn)
            spatial = jnp.clip(spatial, -8.0, 8.0)

        return nn.Dense(self.vocab_size)(spatial)

#Initialize
model        = HyperbolicCodingModel()
key          = jax.random.PRNGKey(0)
dummy_tokens = jnp.ones((2, 64), dtype=jnp.int32)
dummy_depths = jnp.ones((2, 64), dtype=jnp.int32)
params       = model.init(key, dummy_tokens, dummy_depths)

n_params = sum(x.size for x in
               jax.tree_util.tree_leaves(params))
print(f"Parameters: {n_params/1e6:.2f}M")

optimizer    = optax.adam(1e-3)
opt_state    = optimizer.init(params)

params_replicated    = jax.device_put_replicated(
    params,    jax.devices())
opt_state_replicated = jax.device_put_replicated(
    opt_state, jax.devices())

print(f"Replicated across {N_DEVICES} devices")

Parameters: 1.64M
Replicated across 8 devices


In [74]:
import threading
from queue import Queue
from functools import partial

# Threaded prefetch
def prefetch_worker(queue, n_steps,
                    batch_per_device=8, seq_len=2048):
    for step in range(n_steps):
        batch = load_batch(step, batch_per_device, seq_len)
        queue.put(batch)
    queue.put(None)

# pmap train step
@partial(jax.pmap, axis_name="devices")
def train_step_B(params, opt_state, batch):
    def loss_fn(params):
        logits  = model.apply(params,
                               batch["token_ids"],
                               batch["depths"])
        targets = batch["token_ids"][:, 1:]
        logits  = logits[:, :-1, :]
        return optax.softmax_cross_entropy_with_integer_labels(
            logits, targets).mean()

    loss, grads = jax.value_and_grad(loss_fn)(params)
    grads = jax.lax.pmean(grads, axis_name="devices")
    loss  = jax.lax.pmean(loss,  axis_name="devices")

    updates, new_opt_state = optimizer.update(grads, opt_state)
    new_params = optax.apply_updates(params, updates)
    return loss, new_params, new_opt_state

# Start prefetch thread
n_steps      = 5000
batch_queue  = Queue(maxsize=8)
thread       = threading.Thread(
    target=prefetch_worker,
    args=(batch_queue, n_steps),
    kwargs={"batch_per_device": 8, "seq_len": 2048},
    daemon=True
)
thread.start()

print("=" * 50)
print("Experiment B : Hyperbolic, No Stabilization Phase 3 (5000 steps) ")
print("=" * 50)

results_B = []

for step in range(n_steps):
    batch = batch_queue.get()
    if batch is None:
        break

    loss, params_replicated, opt_state_replicated = train_step_B(
        params_replicated,
        opt_state_replicated,
        batch
    )

    loss_scalar = float(loss[0])

    if jnp.isnan(loss[0]):
        print(f"NaN at step {step}")
        break

    results_B.append(loss_scalar)

    if step % 100 == 0:
        print(f"Step {step:4d} | Loss: {loss_scalar:.4f}")

print(f"\nExperiment B done")
print(f"Final loss (avg last 50): {np.mean(results_B[-50:]):.4f}")

with open("/kaggle/working/results_B_3.json", "w") as f:
    json.dump(results_B, f)
print("Saved")

Experiment B : Hyperbolic, No Stabilization Phase 3 (5000 steps) 
Step    0 | Loss: 11.6722
Step  100 | Loss: 4.3709
Step  200 | Loss: 1.6535
Step  300 | Loss: 1.3764
Step  400 | Loss: 1.4249
Step  500 | Loss: 1.2910
Step  600 | Loss: 1.2265
Step  700 | Loss: 1.1363
Step  800 | Loss: 1.0816
Step  900 | Loss: 1.1443
Step 1000 | Loss: 1.0371
Step 1100 | Loss: 0.9931
Step 1200 | Loss: 0.9535
Step 1300 | Loss: 0.9155
Step 1400 | Loss: 0.9905
Step 1500 | Loss: 0.9235
Step 1600 | Loss: 0.8876
Step 1700 | Loss: 0.8701
Step 1800 | Loss: 0.8314
Step 1900 | Loss: 0.9131
Step 2000 | Loss: 0.8529
Step 2100 | Loss: 0.8254
Step 2200 | Loss: 0.8153
Step 2300 | Loss: 0.7849
Step 2400 | Loss: 0.8648
Step 2500 | Loss: 0.8125
Step 2600 | Loss: 0.7809
Step 2700 | Loss: 0.7730
Step 2800 | Loss: 0.7483
Step 2900 | Loss: 0.8232
Step 3000 | Loss: 0.7811
Step 3100 | Loss: 0.7425
Step 3200 | Loss: 0.7393
Step 3300 | Loss: 0.7163
Step 3400 | Loss: 0.7931
Step 3500 | Loss: 0.7639
Step 3600 | Loss: 0.7137
Step 370